In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DecimalType,
    TimestampType
)

EXPECTED_SCHEMA = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("account_id", StringType(), False),
    StructField("merchant_id", StringType(), False),
    StructField("transaction_amount", DecimalType(18, 2), False),
    StructField("currency", StringType(), False),
    StructField("transaction_type", StringType(), False),
    StructField("transaction_status", StringType(), False),
    StructField("transaction_timestamp", TimestampType(), False),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False)
])

In [0]:
def validate_schema(df, expected_schema):
    
    actual_fields = {
        field.name: field
        for field in df.schema.fields
    }
    
    expected_fields = {
        field.name: field
        for field in expected_schema.fields
    }
    
    actual_columns = set(actual_fields.keys())
    expected_columns = set(expected_fields.keys())
    
    missing_columns = sorted(
        expected_columns - actual_columns
    )
    
    unexpected_columns = sorted(
        actual_columns - expected_columns
    )
    
    datatype_mismatches = []
    
    for column_name in sorted(
        expected_columns.intersection(actual_columns)
    ):
        
        expected_type = expected_fields[column_name].dataType
        actual_type = actual_fields[column_name].dataType
        
        if expected_type != actual_type:
            datatype_mismatches.append({
                "column": column_name,
                "expected": str(expected_type),
                "actual": str(actual_type)
            })
    
    return {
        "is_valid": (
            len(missing_columns) == 0
            and len(unexpected_columns) == 0
            and len(datatype_mismatches) == 0
        ),
        "missing_columns": missing_columns,
        "unexpected_columns": unexpected_columns,
        "datatype_mismatches": datatype_mismatches
    }

In [0]:
BRONZE_TABLE = "dbx_fintech_data_platform.bronze.transactions_autoloader_final"

transactions_df = spark.table(BRONZE_TABLE)

transactions_df.printSchema()

In [0]:
TRANSACTION_COLUMNS = [
    "transaction_id",
    "account_id",
    "merchant_id",
    "transaction_amount",
    "currency",
    "transaction_type",
    "transaction_status",
    "transaction_timestamp",
    "created_at",
    "updated_at"
]

transactions_business_df = transactions_df.select(
    *TRANSACTION_COLUMNS
)

In [0]:
transactions_business_df.printSchema()

In [0]:
validation_result = validate_schema(
    transactions_business_df,
    EXPECTED_SCHEMA
)

validation_result

In [0]:
transactions_silver_df = (
    transactions_business_df
    .withColumn(
        "transaction_amount",
        F.col("transaction_amount").cast("decimal(18,2)")
    )
)

In [0]:
transactions_silver_df.printSchema()

In [0]:
validation_result = validate_schema(
    transactions_silver_df,
    EXPECTED_SCHEMA
)

validation_result

In [0]:
def enforce_schema_contract(df, expected_schema):
    
    result = validate_schema(
        df,
        expected_schema
    )
    
    if not result["is_valid"]:
        raise ValueError(
            f"Schema validation failed: {result}"
        )
    
    return df

In [0]:
validated_df = enforce_schema_contract(
    transactions_silver_df,
    EXPECTED_SCHEMA
)